# Preparación de datos — Localidad Riomar (Registraduría + DANE)

Genera, a partir de las fuentes oficiales descargadas, los archivos de entrada de `modelo_participacion_riomar.ipynb`.

Fuentes esperadas en `00_Fuentes/`:
* `MMV_TERRITORIALES2019_ATLÁNTICO.zip` y `MMV_TERRITORIALES2023_ATLÁNTICO.zip` (Registraduría, votos mesa a mesa)
* `2019-AUTORIDADES_TERRITORIALES-ATLANTICO-BARRANQUILLA.csv` y `2023-...csv` (DIVIPOLE: potencial, edad y sexo por puesto)
* `Divipole_Elecciones_Territoritoriales_2023_con_georreferenciación.csv` (coordenadas de puestos)
* `08_Atlantico.zip` (microdatos CNPV 2018, DANE) y `MGN_URB_MANZANA2018geopackage.zip` (cartografía de manzanas 2018)

In [ ]:
import os, glob, zipfile, re, unicodedata
import numpy as np, pandas as pd, geopandas as gpd, pyogrio
from scipy.spatial import cKDTree

BASE = '/content/drive/MyDrive/TesisRiomar'
FUENTES = os.path.join(BASE, '00_Fuentes')
CRS = 'EPSG:9377'
for d in ['01_Cartografia_GIS', '02_Datos_Sociodemograficos_DANE', '03_Datos_Electorales_Registraduria']:
    os.makedirs(os.path.join(BASE, d), exist_ok=True)
F = lambda *p: os.path.join(FUENTES, *p)
O = lambda *p: os.path.join(BASE, *p)

## 1. Votos por mesa (Alcaldía, Localidad 5 Riomar)

In [ ]:
COLS = ['Cod_Depto', 'Depto', 'Cod_Mpio', 'Mpio', 'Cod_Zona', 'Cod_Puesto', 'Puesto', 'Mesa', 'Cod_Comuna', 'Comuna',
        'Cod_Corp', 'Corporacion', 'Cod_Circun', 'Cod_Partido', 'Partido', 'Cod_Candidato', 'Candidato', 'Votos']
def leer_mmv(anio):
    z = [x for x in glob.glob(F('MMV_TERRITORIALES%d*.zip' % anio))][0]
    with zipfile.ZipFile(z) as zz:
        name = [n for n in zz.namelist() if n.lower().endswith('.csv') and 'HASH' not in n.upper()][0]
        with zz.open(name) as fh:
            df = pd.read_csv(fh, sep=None, engine='python', encoding='latin-1', dtype=str)
    df.columns = COLS
    df['Votos'] = pd.to_numeric(df.Votos)
    r = df[df.Mpio.str.contains('BARRANQUILLA', na=False) & df.Corporacion.str.contains('ALCALD', na=False)
           & df.Comuna.str.contains('RIOMAR', na=False)].copy()
    r['Puesto'] = r.Puesto.str.replace('Ã\x91', 'Ñ').str.replace('ÃANZA', 'ÑANZA')
    m = r.groupby(['Cod_Zona', 'Cod_Puesto', 'Puesto', 'Mesa']).Votos.sum().reset_index(name='sufragantes')
    m['anio'] = anio; m['codigo_puesto'] = m.Cod_Zona + '-' + m.Cod_Puesto; m['mesa'] = pd.to_numeric(m.Mesa)
    return m
m19, m23 = leer_mmv(2019), leer_mmv(2023)

## 2. Cruce de puestos 2019–2023 (la Registraduría cambió los códigos de zona)

In [ ]:
def norm(s):
    s = unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode().upper()
    s = re.sub(r'\b(COL|COLEGIO|I\.?E\.?D?|IED|C\.?E\.?D|SCHOOL|SEDE|SD|DE)\b\.?', '', s); s = re.sub(r'[^A-Z ]', ' ', s)
    return ' '.join(s.replace('CONCEJO', 'CONSEJO').replace('LINDON', 'LYNDON').split())
p19 = m19.drop_duplicates('codigo_puesto')[['codigo_puesto', 'Puesto']]; p23 = m23.drop_duplicates('codigo_puesto')[['codigo_puesto', 'Puesto']]
p19['k'] = p19.Puesto.map(norm); p23['k'] = p23.Puesto.map(norm)
rows = []
for _, a in p23.sort_values('codigo_puesto').iterrows():
    hit = p19[(p19.k == a.k) | p19.k.apply(lambda x: x in a.k or a.k in x)]
    if hit.empty and a.codigo_puesto == '99-10':        # La Playa -> I.E. Fundación Pies Descalzos (mismo código)
        hit = p19[p19.codigo_puesto == '99-10']
    rows.append(dict(puesto_id='R%02d' % len(rows), codigo_2023=a.codigo_puesto, nombre_2023=a.Puesto,
                     codigo_2019=hit.codigo_puesto.iloc[0] if len(hit) else '', nombre_2019=hit.Puesto.iloc[0] if len(hit) else '(nuevo en 2023)'))
cw = pd.DataFrame(rows)
print(cw.to_string(index=False))
sin_par = set(p19.codigo_puesto) - set(cw.codigo_2019)
print('Puestos de 2019 sin equivalente en 2023 (se excluyen):', sin_par)
cw.to_csv(O('03_Datos_Electorales_Registraduria', 'cruce_puestos_2019_2023.csv'), index=False)
m19['puesto_id'] = m19.codigo_puesto.map(dict(zip(cw.codigo_2019, cw.puesto_id)))
m23['puesto_id'] = m23.codigo_puesto.map(dict(zip(cw.codigo_2023, cw.puesto_id)))
mm = pd.concat([m19, m23]).dropna(subset=['puesto_id'])

## 3. DIVIPOLE: potencial, edad y sexo de los inscritos por puesto; potencial aproximado por mesa

In [ ]:
DCOLS = ['anio', 'tipo', 'cod_depto', 'depto', 'cod_mpio', 'mpio', 'zona', 'cod_puesto', 'puesto', 'hombres', 'mujeres', 'otros',
         'potencial', 'e18_20', 'e21_25', 'e26_30', 'e31_35', 'e36_40', 'e41_45', 'e46_50', 'e51_55', 'e56_60', 'e60_mas', 'e_sin', 'extranjero']
dv = []
for y in (2019, 2023):
    d = pd.read_csv(glob.glob(F('%d-AUTORIDADES_TERRITORIALES*BARRANQUILLA*.csv' % y))[0], skiprows=2, header=None, names=DCOLS, dtype=str).dropna(subset=['zona'])
    for c in DCOLS[9:]: d[c] = pd.to_numeric(d[c].str.replace('.', '', regex=False))
    d['codigo_puesto'] = d.zona.str.zfill(2) + '-' + d.cod_puesto.str.zfill(2); d['anio'] = y
    dv.append(d)
dv = pd.concat(dv)
q = mm.groupby(['anio', 'puesto_id', 'codigo_puesto']).agg(n_mesas=('mesa', 'size'), sufragantes=('sufragantes', 'sum')).reset_index()
q = q.merge(dv, on=['anio', 'codigo_puesto'], how='left')
assert q.potencial.notna().all()

# Potencial por mesa: cupos iguales y la última mesa con el remanente (cupo mínimo compatible con los sufragantes)
out = []
for (y, pid), g in mm.groupby(['anio', 'puesto_id']):
    pot = int(q[(q.anio == y) & (q.puesto_id == pid)].potencial.iloc[0]); n = len(g); g = g.sort_values('mesa').copy()
    for s in range(int(np.ceil(pot / n)), int(np.ceil(pot / n)) + 60):
        last = pot - (n - 1) * s
        if last < 1: break
        sizes = [s] * (n - 1) + [last]
        if all(np.array(sizes) >= g.sufragantes.values): break
    g['potencial'] = sizes; out.append(g)
mesas = pd.concat(out)[['anio', 'puesto_id', 'mesa', 'potencial', 'sufragantes']]
mesas.to_csv(O('03_Datos_Electorales_Registraduria', 'mesas.csv'), index=False)

Fp = pd.DataFrame({'anio': q.anio, 'puesto_id': q.puesto_id, 'potencial_puesto': q.potencial, 'sufragantes_puesto': q.sufragantes,
                   'p_mujeres': q.mujeres / q.potencial, 'p_18_25': (q.e18_20 + q.e21_25) / q.potencial,
                   'p_26_40': (q.e26_30 + q.e31_35 + q.e36_40) / q.potencial,
                   'p_41_60': (q.e41_45 + q.e46_50 + q.e51_55 + q.e56_60) / q.potencial, 'p_60_mas': q.e60_mas / q.potencial})
Fp.to_csv(O('02_Datos_Sociodemograficos_DANE', 'puestos_features.csv'), index=False)
print(Fp.groupby('anio')[['potencial_puesto', 'sufragantes_puesto']].sum())

## 4. Coordenadas de puestos

In [ ]:
def num(s):
    s = str(s).strip()
    if ',' in s and '.' not in s:
        sign = '-' if s.startswith('-') else ''; parts = s.lstrip('-').split(','); s = sign + parts[0] + '.' + ''.join(parts[1:])
    return float(s)
geo = pd.read_csv(glob.glob(F('Divipole_Elecciones_Territoritoriales_2023*.csv'))[0], dtype=str)
baq = geo[geo.MUNICIPIO == 'BARRANQUILLA'].copy(); baq['lat'] = baq.LATITUD.map(num); baq['lon'] = baq.LONGITUD.map(num)
baq = baq[baq.lat.between(10.85, 11.1) & baq.lon.between(-74.95, -74.7)]
rows = []
for _, r in cw.iterrows():
    g = baq[baq.PUESTO.str.strip() == r.nombre_2023.strip()]
    assert len(g) == 1, r.nombre_2023
    rows.append(dict(anio=2023, puesto_id=r.puesto_id, nombre=r.nombre_2023, lat=g.lat.iloc[0], lon=g.lon.iloc[0]))
    if r.codigo_2019: rows.append(dict(anio=2019, puesto_id=r.puesto_id, nombre=r.nombre_2019, lat=g.lat.iloc[0], lon=g.lon.iloc[0]))
pd.DataFrame(rows).to_csv(O('01_Cartografia_GIS', 'puestos.csv'), index=False)

## 5. Variables censales por manzana (CNPV 2018, microdatos DANE) y área de estudio

Se usan los microdatos del CNPV 2018 para Atlántico (`08_Atlantico_CSV.zip`) y la cartografía MGN 2018 de manzanas
(`MGN_URB_MANZANA2018geopackage.zip`). La llave de unión es `COD_DANE_ANM` = `MANZ_CCNCT`.
El área de estudio es la Localidad Riomar según el código de localidad del censo (`UA1_LOCALIDAD = 01`).
Códigos usados (verificar con el diccionario del CNPV): P_EDADR en grupos quinquenales (1 = 0–4 años, 4 = 15–19,
5 = 20–24, …); P_NIVEL_ANOSR: 8 = universitario, 9 = especialización, maestría o doctorado; VA1_ESTRATO: 1–6.

In [ ]:
LOC_RIOMAR = '01'
zc = zipfile.ZipFile(glob.glob(F('08_Atlantico*.zip'))[0])
inner = [n for n in zc.namelist() if n.endswith('_CSV.zip')]
if inner:
    import io
    zc = zipfile.ZipFile(io.BytesIO(zc.read(inner[0])))
def leer_cnpv(clave, cols):
    name = [n for n in zc.namelist() if clave in n][0]
    with zc.open(name) as fh:
        return pd.read_csv(fh, dtype=str, usecols=cols)
geo_c = leer_cnpv('_MGN_', ['U_MPIO', 'UA_CLASE', 'UA1_LOCALIDAD', 'COD_ENCUESTAS', 'U_VIVIENDA', 'COD_DANE_ANM'])
geo_c = geo_c[(geo_c.U_MPIO == '001') & (geo_c.UA1_LOCALIDAD == LOC_RIOMAR)]
viv = leer_cnpv('_1VIV_', ['U_MPIO', 'COD_ENCUESTAS', 'U_VIVIENDA', 'V_CON_OCUP', 'VA1_ESTRATO'])
per = leer_cnpv('_5PER_', ['U_MPIO', 'COD_ENCUESTAS', 'U_VIVIENDA', 'P_SEXO', 'P_EDADR', 'P_NIVEL_ANOSR'])
key = ['COD_ENCUESTAS', 'U_VIVIENDA']
viv = viv[viv.U_MPIO == '001'].merge(geo_c[key + ['COD_DANE_ANM']], on=key)
per = per[per.U_MPIO == '001'].merge(geo_c[key + ['COD_DANE_ANM']], on=key)
per['edad'] = pd.to_numeric(per.P_EDADR); per['edu'] = pd.to_numeric(per.P_NIVEL_ANOSR, errors='coerce')
viv['estr'] = pd.to_numeric(viv.VA1_ESTRATO, errors='coerce')
ve = viv[(viv.V_CON_OCUP == '1') & viv.estr.between(1, 6)]
E_ = ve.groupby('COD_DANE_ANM').estr.agg(n_viv='size', p_estrato_bajo=lambda s: (s <= 2).mean(),
                                          p_estrato_medio=lambda s: s.between(3, 4).mean(), p_estrato_alto=lambda s: (s >= 5).mean(),
                                          estrato_medio='mean')
ad = per[per.edad >= 5]                                   # 20 años o más
P_ = ad.groupby('COD_DANE_ANM').agg(n_20=('edad', 'size'),
                                     p_20_29=('edad', lambda s: s.between(5, 6).mean()), p_30_44=('edad', lambda s: s.between(7, 9).mean()),
                                     p_45_64=('edad', lambda s: s.between(10, 13).mean()), p_65_mas=('edad', lambda s: (s >= 14).mean()),
                                     p_mujeres_cnpv=('P_SEXO', lambda s: (s == '2').mean()))
U_ = per[(per.edad >= 6) & per.edu.between(1, 10)].groupby('COD_DANE_ANM').edu.agg(p_edu_superior=lambda s: s.isin([8, 9]).mean())
n15 = per[per.edad == 4].groupby('COD_DANE_ANM').size().rename('n_15_19')
V = P_.join(E_, how='left').join(U_, how='left').join(n15, how='left').fillna({'n_15_19': 0})
V['pob_18'] = (V.n_20 + 0.4 * V.n_15_19).round().astype(int)   # 18–19 años ≈ 40 % del grupo 15–19
faltan = V[['p_estrato_bajo', 'p_edu_superior']].isna().sum().to_dict()
print('Manzanas con población adulta:', len(V), '| faltantes:', faltan)
for c in ['p_estrato_bajo', 'p_estrato_medio', 'p_estrato_alto', 'estrato_medio', 'p_edu_superior']:
    V[c] = V[c].fillna(V[c].median())                         # pocas manzanas sin viviendas ocupadas con estrato
z = glob.glob(F('MGN_URB_MANZANA2018*.zip'))[0]
gp = [n for n in zipfile.ZipFile(z).namelist() if n.lower().endswith('.gpkg')][0]
mz = pyogrio.read_dataframe(f'/vsizip/{z}/{gp}', where="MPIO_CCDGO = '08001' AND CLAS_CCDGO = '1'")
area = mz.merge(V.reset_index().rename(columns={'COD_DANE_ANM': 'MANZ_CCNCT'}), on='MANZ_CCNCT', how='inner')
area['manzana_id'] = area.MANZ_CCNCT; area['barrio'] = 'Sección ' + area.COD_SECC.str[-4:]
print(f'Manzanas Localidad Riomar (CNPV 2018): {len(area)} | población de 18 años o más estimada: {int(area.pob_18.sum()):,}')
cols = ['manzana_id', 'pob_18', 'p_estrato_bajo', 'p_estrato_medio', 'p_estrato_alto', 'estrato_medio', 'p_edu_superior',
        'p_20_29', 'p_30_44', 'p_45_64', 'p_65_mas', 'p_mujeres_cnpv']
area[['manzana_id', 'barrio', 'geometry']].to_crs('EPSG:4326').to_file(O('01_Cartografia_GIS', 'manzanas_estudio.gpkg'), driver='GPKG')
area[cols].to_csv(O('02_Datos_Sociodemograficos_DANE', 'manzanas_variables.csv'), index=False)
print(area[cols[1:]].describe().round(3).T[['mean', 'min', 'max']])
print('Archivos generados en', BASE)